[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amistein/ml-study-group/blob/main/notebooks/lesson_003.ipynb)

### Choosing a Loss Function

We've seen two loss functions so far. In lesson 1, we used the `average_absolute_error` function, and in lesson 2 we used the `mean_square_error` function. But we didn't go into why we used different functions, and what the differences between them are. There are many different possible loss functions, and it's worth diving into the differences between some of them, just to demonstrate that the choice of loss function is something that needs to be considered when designing a model, and can affect model outcomes. What is the simplest loss function that you can think of? Well, we could simply calculate the difference betweeen the prediction that the model makes and the actual value for each data point, and sum those up (and maybe take the average) for our total loss. So why don't we just do that? The answer is that there is an obvious problem with this approach. Let's we have one house where the real sale price was 300k and the model predicted 250k, and there is another house with a real sale price of 200k, and for that the model alos predicts 250k. In other words, in one example the loss is positive, and in the other, the loss is negative. So when they're added together, they cancel each other out and the total is 0. So our loss function is telling us that we have a perfect model, when that is definitely not the case!

There are two basic ways to solve this problem. Either take the absolute values of the losses (mean absolute error or **MAE**, what we did in lesson 1), or take the squares of the losses (mean square error or **MSE**, what we did in lesson 2). Taking the squares has a similar effect as taking the absolute value, as the square of any number is positive. The main difference between the two is that MSE penalized larger error values more, so the choice between them depends on what you care about. If you want your model to mostly get things right and you're okay with the occasional big miss, MAE is best. But if you really want to avoid large errors even at the cost of slightly worse average performance, MSE is a better fit because it penalizes those large errors disproportionately.

There is one additional drawback to MSE. And you may have noticed it if you did the exercise in lesson 2. The graph we generated showed a nice U curve, but what were the values on the `y` axis? The text in the top left indicated that the values were on a scale of 1e10, in other words, a value of 3 represents 30,000,000,000. That sounds like a high loss, but it makes sense when we consider that we're squaring every difference. So the loss value does a good job of capturing how good our predictions are, but it would be more useful if it were on the same scale as our price data, so we can get a real world quantification of our loss. So to solve this, in practice people often talk about "root mean square error" or **RMSE**. It just the square root of the MSE, so it has the same effect on the training, but puts the values on the same scale as the data so that we can reason about it more easily.

(MSE is still generally preferred over RMSE for the *actual training*, as it makes the calculations for updating the weights more simple. But we're getting ahead of ourselves.)

One last thing to point out. Different types of problems need differnt types of loss functions. The loss functions we've disscussed here don't work for every application. They are specifically for the type of problem in our current example where we're trying to predict a continuous numerical value (house price). This type of problem is know as a **regression** problem. But what if we wanted to predict a binary value? For example, if we have data about passengers on the titanic, and we want to predict whether or not they survived. This type of problem is known as a **classification** problem, and needs a differnt type of loss function (which we'll talk about when we cover classification).

### **Exercises**
#### Implement the following functions, and ensure the tests pass.

*Reading through the test cases is a good way to make sure you understand what your code is doing.*

In lesson 1, we wrote `average_absolute_error` but it was tied to the housing prediction problem, e.g. it took in houses, weights, and a base price. Let's write more general versions of MAE and MSE that just take two lists of numbers: predictions and actual values. This way we can reuse them with any model, not just our house price predictor.

In [1]:
def mean_absolute_error(predictions, actuals):
    """Compute the mean absolute error between predictions and actuals.

    Args:
        predictions: list of predicted values
        actuals: list of actual values (same length as predictions)
    Returns:
        mean absolute error (a single non-negative number)
    """
    # YOUR CODE HERE
    return sum(abs(p - a) for p, a in zip(predictions, actuals)) / len(actuals)
    raise NotImplementedError()


# --- Tests (do not modify) ---
# Errors of +50k and -50k: raw average would be 0, but MAE should be 50k
_mae1 = mean_absolute_error([350000, 250000], [300000, 300000])
assert _mae1 == 50000, f"Expected 50000, got {_mae1}"

# Perfect predictions: MAE should be 0
_mae2 = mean_absolute_error([300000, 400000], [300000, 400000])
assert _mae2 == 0, f"Expected 0, got {_mae2}"

# All predictions off by the same amount
_mae3 = mean_absolute_error([110, 220, 330], [100, 200, 300])
assert _mae3 == 20, f"Expected 20, got {_mae3}"

print("✅ All tests passed!")

✅ All tests passed!


In [2]:
def mean_squared_error(predictions, actuals):
    """Compute the mean squared error between predictions and actuals.

    Args:
        predictions: list of predicted values
        actuals: list of actual values (same length as predictions)
    Returns:
        mean squared error (a single non-negative number)
    """
    # YOUR CODE HERE
    return sum((p-a) ** 2 for p, a in zip(predictions,actuals))/len(actuals)
    raise NotImplementedError()


# --- Tests (do not modify) ---
# Same scenario: errors of +50k and -50k
_mse1 = mean_squared_error([350000, 250000], [300000, 300000])
assert _mse1 == 2500000000, f"Expected 2500000000, got {_mse1}"

# Perfect predictions
_mse2 = mean_squared_error([300000, 400000], [300000, 400000])
assert _mse2 == 0, f"Expected 0, got {_mse2}"

# One small error and one big error
_mse3 = mean_squared_error([105, 200], [100, 100])
assert _mse3 == 5012.5, f"Expected 5012.5, got {_mse3}"

# Compare: MAE treats these equally, MSE does not
# Two errors of 50 vs. one error of 0 and one of 100
_even = mean_squared_error([150, 250], [100, 200])   # errors: 50, 50
_uneven = mean_squared_error([100, 300], [100, 200]) # errors: 0, 100
assert _even == 2500, f"Expected 2500, got {_even}"
assert _uneven == 5000, f"Expected 5000, got {_uneven}"
assert _uneven > _even, "MSE should penalize the larger error more"

print("✅ All tests passed!")

✅ All tests passed!



Now let's see how the choice of loss function affects which weights we prefer. We'll try different weight sets with housing data, but this time we'll evaluate them with both MAE and MSE to see if they agree on which weights are best.

In [7]:
import math

# Prediction function from lesson 1
def predict(features, weights, base_price):
    return base_price + sum(f * w for f, w in zip(features, weights))

houses = [
    [1500, 3, 2, 10],  # sqft, beds, baths, age
    [2000, 4, 3, 20],
    [800, 2, 1, 40],
    [2500, 5, 3, 5],
]
actual_prices = [300000, 400000, 150000, 500000]
base_price = 50000


def find_best_weights(houses, actual_prices, weight_sets, base_price, loss_fn):
    """Find the weight set with the lowest loss.

    Args:
        houses: list of feature lists
        actual_prices: list of actual sale prices
        weight_sets: list of weight lists to try
        base_price: base price for predictions
        loss_fn: a function that takes (predictions, actuals) and returns a loss value
    Returns:
        tuple of (best_weights, best_loss)
    """
    # YOUR CODE HERE
    # Hint: for each weight set, generate predictions for all houses,
    # then pass the predictions and actual prices to loss_fn.
    # Keep track of which weight set gives the lowest loss.
    best_weights = weight_sets[0]
    best_loss = math.inf
    for w in weight_sets:
      predictions = [predict(h, w, base_price) for h in houses]
      loss = loss_fn(predictions, actual_prices)
      if loss < best_loss:
        best_loss = loss
        best_weights = w
    return (best_weights, best_loss)

    raise NotImplementedError()


# --- Tests (do not modify) ---
weight_sets = [
    [100, 10000, 5000, -3000],
    [150, 20000, 10000, -2000],
    [80, 15000, 8000, -1000],
]

_best_mae, _loss_mae = find_best_weights(houses, actual_prices, weight_sets, base_price, mean_absolute_error)
_best_mse, _loss_mse = find_best_weights(houses, actual_prices, weight_sets, base_price, mean_squared_error)

assert _best_mae == [150, 20000, 10000, -2000], f"MAE best weights wrong: {_best_mae}"
assert _best_mse == [150, 20000, 10000, -2000], f"MSE best weights wrong: {_best_mse}"

print(f"Best weights (MAE): {_best_mae}, loss = {_loss_mae:,.0f}")
print(f"Best weights (MSE): {_best_mse}, loss = {_loss_mse:,.0f}")

# Now try a case where MAE and MSE disagree!
# Weight set A nails one house (only $7,500 off) but badly misses another ($67,500 off).
# Weight set B is more consistent — no single prediction is great, but none are terrible.
tricky_weight_sets = [
    [100, 20000, 12500, -1000],  # has one near-perfect prediction, but also a big miss
    [100, 22500, 12500, -1500],  # more evenly distributed errors
]

_tricky_mae, _ = find_best_weights(houses, actual_prices, tricky_weight_sets, base_price, mean_absolute_error)
_tricky_mse, _ = find_best_weights(houses, actual_prices, tricky_weight_sets, base_price, mean_squared_error)

assert _tricky_mae == [100, 20000, 12500, -1000], f"Tricky MAE wrong: {_tricky_mae}"
assert _tricky_mse == [100, 22500, 12500, -1500], f"Tricky MSE wrong: {_tricky_mse}"
assert _tricky_mae != _tricky_mse, "MAE and MSE should disagree here!"

print(f"\nWith tricky weight sets:")
print(f"MAE prefers: {_tricky_mae}")
print(f"MSE prefers: {_tricky_mse}")
print("They disagree! The loss function you choose affects which weights win.")
print("✅ All tests passed!")

Best weights (MAE): [150, 20000, 10000, -2000], loss = 27,500
Best weights (MSE): [150, 20000, 10000, -2000], loss = 937,500,000

With tricky weight sets:
MAE prefers: [100, 20000, 12500, -1000]
MSE prefers: [100, 22500, 12500, -1500]
They disagree! The loss function you choose affects which weights win.
✅ All tests passed!
